# Complexidade e busca

*Rode uma célula com ▶ (ou Shift+Enter): o Sphinx compila e roda a célula no seu computador. Mude o código e rode de novo: este notebook é a sua cópia.*

**Depois desta unidade você consegue:**
- Prever, pelos limites da entrada, se uma solução cabe no limite de tempo
- Trocar um laço aninhado por um conjunto de hash, ordenação ou dois ponteiros
- Achar fronteiras em dados ordenados com busca binária e contar os valores de um intervalo
- Fazer busca binária na resposta quando testar um candidato é fácil, mas achá-lo direto é difícil

## Em resumo

Limites de tempo transformam o Big O num orçamento. Um computador faz mais ou menos **10⁸ passos simples por segundo**, então leia os limites da entrada antes de escrever código: com n até 200.000, um laço sobre todos os pares dá 4 × 10¹⁰ passos (minutos), enquanto O(n log n) dá uns 3,5 milhões (milissegundos). Como regra prática, n ≤ 5.000 permite O(n²), e n ≤ 500.000 pede O(n log n) ou melhor.

**Rode e olhe as colunas:** quando n dobra, um laço faz o dobro do trabalho, e um laço dentro de outro faz quatro vezes mais.

In [ ]:
for (int n = 1000; n <= 16000; n *= 2) {
    long um = 0;
    long aninhado = 0;
    for (int i = 0; i < n; i++) {
        um++;
    }
    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            aninhado++;
        }
    }
    IO.println(String.format("n = %,6d   um laço: %,7d   laço dentro de laço: %,12d", n, um, aninhado));
}

A maioria das correções troca um **laço aninhado** por uma ferramenta melhor:

- **Um conjunto de hash** responde "já vi isto antes?" em tempo constante, em média.
- **Ordenar** coloca valores iguais ou próximos lado a lado, e uma passada compara vizinhos.
- **Dois ponteiros** percorrem um array ordenado pelas duas pontas, ou como início e fim de uma janela, cada um andando num só sentido.

**Cronometre os dois jeitos** de achar ids repetidos. Depois troque `n` para 40.000: qual tempo dobra, e qual fica quatro vezes maior?

In [ ]:
int n = 20_000;
int[] ids = new Random(1).ints(n, 0, 1_000_000_000).toArray();

long inicio = System.nanoTime();
int repetidosLento = 0;
for (int i = 0; i < n; i++) {
    for (int j = i + 1; j < n; j++) {
        if (ids[i] == ids[j]) {
            repetidosLento++;
        }
    }
}
long meio = System.nanoTime();
Set<Integer> vistos = new HashSet<>();
int repetidosRapido = 0;
for (int id : ids) {
    if (!vistos.add(id)) {
        repetidosRapido++;
    }
}
long fim = System.nanoTime();

IO.println("laço dentro de laço: " + (meio - inicio) / 1_000_000 + " ms, repetidos: " + repetidosLento);
IO.println("HashSet:             " + (fim - meio) / 1_000_000 + " ms, repetidos: " + repetidosRapido);

A **busca binária** faz mais que achar um valor. Em dados ordenados ela acha **fronteiras**, como o primeiro índice com valor ≥ x, e duas fronteiras contam tudo o que está num intervalo:

![O primeiro valor ≥ 4 e o primeiro valor > 4 num array ordenado](boundaries.pt-br.svg)

A próxima célula começa com `// shared`, então ela define `lowerBound` para todas as células abaixo. Rode-a, depois a que a usa.

In [ ]:
// shared: as células abaixo podem chamar lowerBound
/** O primeiro índice com valor >= x, ou a.length se não houver. */
int lowerBound(int[] a, long x) {
    int lo = 0;
    int hi = a.length;
    while (lo < hi) {
        int mid = (lo + hi) >>> 1;
        if (a[mid] < x) {
            lo = mid + 1;
        } else {
            hi = mid;
        }
    }
    return lo;
}

In [ ]:
int[] a = {2, 4, 4, 4, 7, 9};
IO.println("primeiro >= 4: índice " + lowerBound(a, 4));
IO.println("primeiro > 4:  índice " + lowerBound(a, 5)); // "primeiro > 4" é "primeiro >= 5"
IO.println("quantos 4: " + (lowerBound(a, 5) - lowerBound(a, 4)));
IO.println("primeiro >= 10: índice " + lowerBound(a, 10) + " (depois do fim)");
// Experimente: quantos valores estão entre 3 e 8, inclusive?

Ela também busca uma **resposta**: quando "c funciona?" é não até certo valor e sim dali em diante, a busca binária acha o primeiro sim em uns 30 testes, mesmo entre 1 e 10⁹.

**Busca binária na resposta:** o menor x com x × x ≥ alvo. Experimente outros alvos.

In [ ]:
long alvo = 2_000_000_000_000L;
long lo = 0;
long hi = 2_000_000;
int testes = 0;
while (lo < hi) {
    long mid = (lo + hi) / 2;
    testes++;
    if (mid * mid >= alvo) {
        hi = mid;
    } else {
        lo = mid + 1;
    }
}
IO.println("menor x: " + lo + ", achado em " + testes + " testes");

O erro mais comum é errar por um: decida se `hi` é o último índice ou um depois dele, e mantenha essa escolha em todo lugar.

## Leituras

- [Time Complexity](https://usaco.guide/bronze/time-comp) (USACO Guide, 15 min): A tabela em "Common Complexities and Constraints": ela transforma um limite da entrada na complexidade mais lenta que ainda cabe. Os exemplos podem aparecer em Java: escolha a linguagem nas configurações do site.
- [Binary Search](https://usaco.guide/silver/binary-search) (USACO Guide, 20 min): Busca binária numa função de sim ou não (o primeiro x em que f(x) é verdadeira) e os três erros comuns: errar por um, limites negativos e estouro. O primeiro índice com valor ≥ x é a mesma busca, com f(i) = "a[i] ≥ x". Pule as listas de problemas.
- [Two Pointers](https://usaco.guide/silver/two-pointers) (USACO Guide, 12 min): "Sum of Two Values" (ponteiros vindo das duas pontas de um array ordenado) e "Sliding Window" (um início e um fim que só andam para a direita, numa animação). Pule as listas de problemas.

## Verifique

1. Com n ≤ 100.000, uma solução O(n²) cabe num limite de um segundo?
2. Em `2 4 4 4 7 9`, qual é o primeiro índice com valor ≥ 5?
3. Por que dá para fazer busca binária na menor capacidade de caminhão que funciona, se nenhum array guarda as capacidades?

## Desafios

Resolva os desafios da unidade aqui mesmo: cada célula começa com `// sphinx: challenge …`, e rodá-la roda **todos** os testes do desafio, inclusive os ocultos. Passar conta no seu progresso na barra lateral do Sphinx, como o Submit. A descrição completa, com exemplos e dicas, abre pela barra lateral.

### Ingressos repetidos · Fácil

In [ ]:
// sphinx: challenge repeated-tickets
void main() {
    int n = Integer.parseInt(IO.readln().trim());
    String[] parts = IO.readln().trim().split(" +");
    int[] ids = new int[n];
    for (int i = 0; i < n; i++) {
        ids[i] = Integer.parseInt(parts[i]);
    }

    // TODO: conte quantos ids diferentes aparecem mais de uma vez, rápido o bastante para 200.000 ingressos
}

### Janela de log · Médio

In [ ]:
// sphinx: challenge log-window
void main() {
    int n = Integer.parseInt(IO.readln().trim());
    String[] parts = IO.readln().trim().split(" +");
    int[] times = new int[n];
    for (int i = 0; i < n; i++) {
        times[i] = Integer.parseInt(parts[i]);
    }
    int q = Integer.parseInt(IO.readln().trim());
    long total = 0;
    int busiest = 0;
    for (int i = 0; i < q; i++) {
        String[] window = IO.readln().trim().split(" +");
        int a = Integer.parseInt(window[0]);
        int b = Integer.parseInt(window[1]);

        // TODO: conte os instantes com a <= t <= b usando busca binária, e atualize total e busiest
    }
    IO.println("total: " + total);
    IO.println("busiest: " + busiest);
}

### Capacidade do caminhão · Médio

In [ ]:
// sphinx: challenge truck-capacity
void main() {
    String[] first = IO.readln().trim().split(" +");
    int n = Integer.parseInt(first[0]);
    int d = Integer.parseInt(first[1]);
    String[] parts = IO.readln().trim().split(" +");
    int[] weights = new int[n];
    for (int i = 0; i < n; i++) {
        weights[i] = Integer.parseInt(parts[i]);
    }

    // TODO: faça busca binária na menor capacidade que despacha todos os pacotes em no máximo d dias
}

### Limitador de requisições · Difícil

In [ ]:
// sphinx: challenge rate-limiter
void main() {
    String[] first = IO.readln().trim().split(" +");
    int n = Integer.parseInt(first[0]);
    int w = Integer.parseInt(first[1]);
    int limit = Integer.parseInt(first[2]);
    String[] parts = IO.readln().trim().split(" +");
    int[] times = new int[n];
    for (int i = 0; i < n; i++) {
        times[i] = Integer.parseInt(parts[i]);
    }

    // TODO: ordene os instantes, ache a janela mais movimentada com dois ponteiros e depois simule o limitador
}